In [ ]:
# ============================================================
# 🥷 AIBO 起動 [Cell-deps] ── deps 確定(PuLID 有効化 + numpy 単一固定で再起動を最小化)
#   PuLID(PuLIDFluxPipeline)は insightface→onnxruntime に依存。onnxruntime が import 不可だと
#   build 時に素 FluxPipeline へ縮退し、生成 Pass1 が id_image エラーで落ちる(05 は pipe_base 型に
#   関係なく id_image を渡す)。→ ここで onnxruntime を import 成功版で確定する。
#   numpy は「AIBO スタック(nunchaku/insightface/facexlib/basicsr)が同時に満たす単一固定版」に
#   exact pin し、PIP_CONSTRAINT で後続 build を含む全 pip が numpy を動かせないようにする
#   = ダウングレード/再起動の毎回往復を撲滅。再起動は「既ロード numpy の ABI が固定版と食い違う
#   時だけ・最大1回」。
# ============================================================
import os, sys, subprocess, importlib.metadata as _md

# ── ★単一固定版(変更は司令部判断・ここ 1 箇所だけ)─────────────────────────────
#   観測: 旧 `numpy>=2.1.0` レンジ + 末尾 `--upgrade` は resolver が 2.4.6 へ毎回ダウングレード
#   → ABI 再起動が毎回挟まっていた。スタックが受理する 2.4.6 に exact pin して非決定性を消す。
#   もし pip conflict(あるパッケージが別版を要求)が出たら、この 1 行を全スタックが満たす版へ。
AIBO_NUMPY = "2.4.6"

_MARK = "/content/.aibo_deps_tried"      # 「導入+(必要なら)再起動 済み」マーカー(restart 無限ループ防止)
_CONSTR = "/content/aibo_constraints.txt"
with open(_CONSTR, "w") as _f:           # build 含む後続の全 pip install に numpy を exact 固定
    _f.write("numpy==" + AIBO_NUMPY + "\n")
os.environ["PIP_CONSTRAINT"] = _CONSTR

def _ver(p):
    try:
        return _md.version(p)
    except Exception:
        return None

def _numpy_ok():
    return _ver("numpy") == AIBO_NUMPY    # exact 一致のみ OK(レンジ判定をやめ非決定性を排除)

def _onnx_import_ok():
    """onnxruntime の import を「クリーンな別プロセス」で検証(現プロセスの壊れ状態に依存しない)。"""
    r = subprocess.run([sys.executable, "-c", "import onnxruntime as o; print(o.__version__)"],
                       capture_output=True, text=True)
    return (r.returncode == 0), (r.stdout.strip() if r.returncode == 0 else (r.stderr.strip()[-200:] or "import 失敗"))

_ok, _info = _onnx_import_ok()
if _ok and _numpy_ok():
    print("✅ [Cell-deps] OK(onnxruntime " + str(_info) + " / numpy " + str(_ver("numpy"))
          + " 固定 / PIP_CONSTRAINT 有効)・再起動不要 → [Cell-run] へ", flush=True)
elif os.path.exists(_MARK):
    # 既に1回 導入+再起動したのに まだ NG = この環境では解決不能 → FATAL([Cell-run] に進ませない)
    raise RuntimeError("[Cell-deps] FATAL: deps 確定不可(onnxruntime import=" + str(_ok) + ": " + str(_info)
                       + " / numpy=" + str(_ver("numpy")) + " 目標 " + AIBO_NUMPY + ")。再試行後も NG → 司令部へ。"
                       "(pip conflict なら AIBO_NUMPY を全スタックが満たす版に変える)")
else:
    print("📦 [Cell-deps] 導入: onnxruntime(gpu→cpu)+ insightface/facexlib + numpy==" + AIBO_NUMPY + " 固定", flush=True)
    open(_MARK, "w").write("tried")      # ★ 再起動ループ防止(この行以降で最大1回だけ os.kill)
    # 競合除去: onnxruntime と onnxruntime-gpu の同居は import 破損の温床 → 両方消してから1つだけ入れる
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime", "onnxruntime-gpu"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime-gpu"], check=False)
    _ok, _info = _onnx_import_ok()
    if not _ok:
        print("   ⚠️ onnxruntime-gpu が import 不可(" + str(_info)[:80] + ")→ CPU 版へ(insightface は CPU 可)", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime-gpu"], check=False)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime"], check=False)
        _ok, _info = _onnx_import_ok()
    # insightface / facexlib(numpy は PIP_CONSTRAINT で exact 固定・onnxruntime は上で確定済)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "insightface", "facexlib"], check=False)
    # numpy を単一固定版へ exact 設置(--no-deps で副作用無し・== なので resolver 非依存=決定的)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "numpy==" + AIBO_NUMPY], check=False)
    print("   onnxruntime import = " + str(_ok) + " (" + str(_info)[:60] + ") / numpy = " + str(_ver("numpy")), flush=True)
    if not _ok:
        raise RuntimeError("[Cell-deps] FATAL: onnxruntime が gpu/cpu とも import 不可: " + str(_info)[:200])
    if not _numpy_ok():
        raise RuntimeError("[Cell-deps] FATAL: numpy=" + str(_ver("numpy")) + " を " + AIBO_NUMPY
                           + " に固定できない。pip ログの conflict を確認(該当パッケージを前出し or AIBO_NUMPY 調整)")
    # ── 再起動は「既ロード numpy の ABI が固定版と食い違う時だけ・最大1回・ideally 0」──
    _loaded = sys.modules.get("numpy")
    _loaded_ver = getattr(_loaded, "__version__", None) if _loaded is not None else None
    if _loaded is not None and _loaded_ver != AIBO_NUMPY:
        print("🔁 [Cell-deps] 既ロード numpy " + str(_loaded_ver) + " ≠ 固定 " + AIBO_NUMPY
              + " → C 拡張 ABI 整合のため 1 回だけ再起動。再起動後 [Cell-deps] を再実行(緑表示)→ [Cell-run] へ。", flush=True)
        sys.stdout.flush()
        os.kill(os.getpid(), 9)          # ★ 最大1回だけ self-restart
    else:
        print("✅ [Cell-deps] numpy 未ロード or 既に " + AIBO_NUMPY + " → 再起動 0 回で確定 → [Cell-run] へ", flush=True)


In [ ]:
# ============================================================
# 🥷 AIBO 起動 [Cell-run] ── model warm → CLEAN 配置 → build → serve(+ ログ蛇口)
#   旧 Cell B/C/D を 1 セルに統合(運用 2 クリック化: [Cell-deps] → [Cell-run])。
#   各フェーズの開始/成否ログは残す。再実行で OOM しないよう先頭で冪等ガード。
#   末尾で /content/aibo_gen.log のログ蛇口を自動 attach(self-test で配線確認)。
# ============================================================
# ── 早期冪等ガード: FastAPI 既起動なら全 phase skip(再 build/二重積み OOM を回避)──
import os as _os0
print("🟢 [Cell-run] PHASE 0: 冪等ガード(既起動チェック)", flush=True)
try:
    import requests as _rq0
    _resp0 = _rq0.get("http://localhost:8000/api/system/status", timeout=2)
    if _resp0.status_code == 200 and _resp0.json().get("orchestrator_attached"):
        print("✅ [Cell-run] FastAPI 既起動 → 全 phase skip(冪等・再 build しない)", flush=True)
        _ef0 = "/content/.env.local.latest.txt"
        if _os0.path.exists(_ef0):
            print(open(_ef0).read().strip(), flush=True)
        raise SystemExit("[Cell-run] 既起動につき終了(冪等)")
except SystemExit:
    raise
except Exception as _e0:
    # 接続不可 = サーバ未起動 = 正常フロー(silent にせず明示してから下へ進む)
    print("ℹ️ [Cell-run] FastAPI 未起動(" + type(_e0).__name__ + ")→ 通常フロー(model→CLEAN→build→serve)", flush=True)

print("\n🟢 [Cell-run] PHASE 1: モデル warm(HF Hub → /content NVMe)", flush=True)

# ============================================================
# 🥷 AIBO 起動 [Cell B] ── モデル warm(HF Hub → /content NVMe 直 DL)
#   Drive FUSE 越しの巨大 safetensors 読込(=truncation→外国人化)を構造的に回避。
#   既 DL 分は resume/skip。code(=CLEAN)・data(=Drive)・models(=HF) の分離は不変。
# ============================================================
import os, sys, importlib, subprocess, shutil, time

# Phase 0: import warm-up は無効化 (VRAM OOM の原因となるため)
# torch/nunchaku を background thread で import すると CUDA コンテキスト初期化が
# 競合し、main thread のメモリプール確保が失敗するケースを確認。
# 4 秒の最適化を捨てて VRAM 安定性を優先。

# ─── Phase 1: Drive mount + HF cache ─────────────────────────
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive", force_remount=False)
# 🆕 CASE-A (RECON-002 第2段 · 案A): モデルは HF Hub から /content(NVMe)へ直 DL する。
# Drive FUSE 越しの巨大 safetensors 読み込み(=truncation→外国人化)を構造的に発生させない。
# Drive はコード(AIBO_ROOT)の置き場としてのみ使い、モデルの一次ソースからは外す。
GDRIVE_HF_CACHE = "/content/drive/MyDrive/aibo_hf_cache"   # sync_back_to_drive() 用に定義のみ保持
CONTENT_HF_CACHE = "/content/aibo_hf_cache"
# 🛡️ FIX-1 (IMPL-003 / H1 対策): /content/aibo_hf_cache が「Drive を指す stale symlink」だと
#   直 DL が Drive に書かれ、ロードで FUSE mmap → OSError errno 19(ENODEV)で落ちる。
#   symlink なら必ず除去してから実ディレクトリを作り、Drive 非経由をアサートする(silent fail 禁止)。
if os.path.islink(CONTENT_HF_CACHE):
    os.unlink(CONTENT_HF_CACHE)          # 旧 notebook が作った Drive 直結 symlink を除去
elif os.path.isdir(CONTENT_HF_CACHE) and os.path.realpath(CONTENT_HF_CACHE).startswith("/content/drive"):
    raise RuntimeError(f"[CASE-A] HF cache が Drive 配下: {os.path.realpath(CONTENT_HF_CACHE)}")
os.makedirs(CONTENT_HF_CACHE, exist_ok=True)
# ★ ガード: 用意した cache が Drive を指していないことを起動時にアサート
_real_cache = os.path.realpath(CONTENT_HF_CACHE)
if _real_cache.startswith("/content/drive"):
    raise RuntimeError(f"[CASE-A] FATAL: HF cache が Drive 上 ({_real_cache})。NVMe 直 DL に失敗。")
print(f"✅ [CASE-A] HF cache = {_real_cache} (NVMe・Drive 非経由を確認)")

# 🗑️ CASE-A で撤去した事故経路:
#   - tar pipe(CACHE_SYNC_TIMEOUT=180)による Drive→/content 一括コピー
#   - timeout 時の shutil.rmtree(CONTENT_HF_CACHE)+os.symlink(GDRIVE→CONTENT)(=Drive 直結)
#   これらが「180s で部分コピー→直結 symlink→FUSE 越し巨大読み込み→truncation」の確定的事故経路だった。

# /root/.cache/huggingface → /content/aibo_hf_cache symlink
# (env 変数を無視する library 対策、全 HF DL を NVMe に固定。Drive ではなく NVMe を指す)
ROOT_HF = "/root/.cache/huggingface"
os.makedirs("/root/.cache", exist_ok=True)
if os.path.islink(ROOT_HF) or os.path.exists(ROOT_HF):
    subprocess.run(["rm", "-rf", ROOT_HF], check=False)
os.symlink(CONTENT_HF_CACHE, ROOT_HF)
print(f"✅ HF cache symlink: {ROOT_HF} → {CONTENT_HF_CACHE}")

# env 変数も /content (NVMe) を指す
for k in ["HF_HOME", "TRANSFORMERS_CACHE", "HUGGINGFACE_HUB_CACHE", "HF_HUB_CACHE"]:
    os.environ[k] = CONTENT_HF_CACHE
# 🆕 CASE-A: HF_HUB_ENABLE_HF_TRANSFER は deprecated 警告が出るため設定しない。
#   高速 DL は hf_xet(既定・huggingface_hub>=0.32)に委ねる。
#   hf_xet の稀なサイズ一致破損は C0 検証ゲート(IMPL-001)が from_pretrained 直前に捕捉する前提。
print(f"✅ HF cache → {CONTENT_HF_CACHE} (NVMe)")

# ─── Phase 1.4: HF token 注入 (userdata 経由・コードに直書きしない) ───
# gated 2 repo(FLUX.1-dev / FLUX.1-Redux-dev)の DL に HF token が必須(G0 調査)。
try:
    from google.colab import userdata
    _hf_tok = userdata.get('HF_TOKEN')   # 既存の NGROK_AUTH_TOKEN と同じ Colab Secrets 仕組み
except Exception as _e:
    _hf_tok = None
    print(f"⚠️ [CASE-A] userdata.get('HF_TOKEN') 取得失敗: {type(_e).__name__}: {_e}")
if _hf_tok:
    os.environ['HF_TOKEN'] = _hf_tok        # 値はログに出さない
    try:
        from huggingface_hub import login
        login(token=_hf_tok)
        print("✅ [CASE-A] HF login OK (token は非表示)")
    except Exception as _e:
        print(f"⚠️ [CASE-A] HF login 失敗: {type(_e).__name__}: {_e}")
else:
    # 握りつぶさない: token 不在を明示警告(gated DL は 401/403 で失敗する)
    print("⚠️ [CASE-A] HF_TOKEN 未設定。gated モデル(FLUX.1-dev/Redux)の DL は失敗します")

# ─── Phase 1.5: HF Hub → /content 直 DL (snapshot_download 一次化) ───
# default cache-dir 方式(local_dir 指定しない=巨大単一ファイルの resume 堅牢性のため · RECON-002 F3)。
# 出口は C0 検証ゲート(IMPL-001)が各 from_pretrained 直前に守る。
from huggingface_hub import snapshot_download
# (repo_id, allow_patterns, ignore_patterns) — None は無指定(allow=None で repo 丸ごと / ignore=None で除外なし)。
_HF_REPOS = [
    # gated (HF_TOKEN 必須)
    ("black-forest-labs/FLUX.1-dev", None, ["flux1-dev.safetensors", "transformer/*"]),  # IMPL-009: bf16 transformer 2塊を除外(Nunchaku INT4 使用)
    ("black-forest-labs/FLUX.1-Redux-dev", None, None),
    # public (token 不要)
    # 🛡️ FIX-2 (IMPL-003 / H2 対策): nunchaku は int4 をピン。A100(Ampere)= int4(get_precision 準拠)。
    #   fp4 は Blackwell 専用なので落とさない。無駄 DL(fp4 ~6.5GB)削減 + DL/ロードの precision 整合。
    ("nunchaku-tech/nunchaku-flux.1-dev",          # INT4 transformer 本体(307→nunchaku-ai に自動追従)
     ["svdq-int4_r32-flux.1-dev.safetensors", "*.json", "README*"], None),
    ("mit-han-lab/svdq-int4-flux.1-fill-dev", None, None),       # Fill transformer
    ("Shakker-Labs/FLUX.1-dev-ControlNet-Union-Pro-2.0", None, None),
    ("XLabs-AI/flux-ip-adapter", None, None),
    ("guozinan/PuLID", None, None),
    ("ByteDance/Hyper-SD", ["Hyper-FLUX.1-dev-8steps-lora.safetensors"], None),  # IMPL-009: 8steps LoRA 単一化
]
print(f"\n📥 [CASE-A] HF Hub → /content 直 DL 開始 ({len(_HF_REPOS)} repo)")
_t_dl = time.time()
for _repo, _allow, _ignore in _HF_REPOS:
    try:
        snapshot_download(repo_id=_repo, allow_patterns=_allow, ignore_patterns=_ignore)   # allow/ignore=None は無指定。既ロード分は resume/skip。実体は /content(NVMe)
        print(f"  ✅ [CASE-A] OK: {_repo}")
    except Exception as _e:
        # 握りつぶさない: gated は token 無/未同意で 401/403。明示 fail-fast(縮退に逆戻りさせない)。
        print(f"  ❌ [CASE-A] FAIL: {_repo} :: {type(_e).__name__}: {_e}")
        raise
print(f"✅ [CASE-A] 直 DL 完了: {time.time()-_t_dl:.1f}s")

# ─── Phase 1.6: insightface antelopev2 配置(PuLID 縮退の真因対策)───
#   PuLID(03 公式 pulid / 04 nunchaku PuLIDFluxPipeline)も 08 顔検出も insightface antelopev2 を使う。
#   insightface の antelopev2 自動 DL は上流で壊れている(release 404)ため、自前で取得して
#   「探されうる 3 root すべて」に配置する。無いと build 時に PuLID が素 FluxPipeline へ縮退し、
#   生成 Pass1 が `FluxPipeline.__call__() got 'id_image'` で落ちる。
import shutil as _sh
_ANTE_FILES = ["1k3d68.onnx", "2d106det.onnx", "genderage.onnx", "glintr100.onnx", "scrfd_10g_bnkps.onnx"]
_HF_HOME = os.environ.get("HF_HOME", "/content/aibo_hf_cache")
_ANTE_CANON = os.path.join(_HF_HOME, "insightface", "models", "antelopev2")   # 08_face_refiner の root
# 探されうる他 root: insightface 既定(~/.insightface)/ 公式 pulid の root='.'(CWD=/content)
_ANTE_MIRRORS = ["/root/.insightface/models/antelopev2", "/content/models/antelopev2"]

def _ante_have(_d):
    return all(os.path.exists(os.path.join(_d, _f)) and os.path.getsize(os.path.join(_d, _f)) > 0 for _f in _ANTE_FILES)

print("\n🧩 [antelopev2] PuLID 顔検出モデルを配置(insightface 自動 DL は上流破損のため自前取得)", flush=True)
if not _ante_have(_ANTE_CANON):
    os.makedirs(_ANTE_CANON, exist_ok=True)
    _got = False
    # 1) Drive の所定パスにあれば最優先(オフライン保険)
    for _dsrc in ("/content/drive/MyDrive/顔/antelopev2",
                  "/content/drive/MyDrive/aibo_models/antelopev2",
                  "/content/drive/MyDrive/aibo_hf_cache/insightface/models/antelopev2"):
        if _ante_have(_dsrc):
            for _f in _ANTE_FILES:
                _sh.copy2(os.path.join(_dsrc, _f), os.path.join(_ANTE_CANON, _f))
            print(f"  ✅ [antelopev2] Drive から配置: {_dsrc}", flush=True)
            _got = True
            break
    # 2) HF Hub から取得(5 onnx を flat に)
    if not _got:
        from huggingface_hub import hf_hub_download
        for _repo in ("DIAMONIK7777/antelopev2",):
            try:
                for _f in _ANTE_FILES:
                    _p = hf_hub_download(repo_id=_repo, filename=_f)
                    _sh.copy2(_p, os.path.join(_ANTE_CANON, _f))
                print(f"  ✅ [antelopev2] HF から配置: {_repo}", flush=True)
                _got = True
                break
            except Exception as _e:
                print(f"  ⚠️ [antelopev2] HF {_repo} 失敗: {type(_e).__name__}: {_e}", flush=True)
    if not _ante_have(_ANTE_CANON):
        raise RuntimeError("[antelopev2] FATAL: .onnx を配置できない。HF が不可なら Drive の所定パス"
                           "(/content/drive/MyDrive/顔/antelopev2/)に 5 つの .onnx を置いて再実行。"
                           " 必要: " + ", ".join(_ANTE_FILES))
else:
    print(f"  ✅ [antelopev2] 既配置: {_ANTE_CANON}", flush=True)
# 3) 他 root へミラー(nunchaku/公式 pulid がどの root を見ても見つかるように)
for _mroot in _ANTE_MIRRORS:
    if not _ante_have(_mroot):
        os.makedirs(os.path.dirname(_mroot), exist_ok=True)
        try:
            if os.path.islink(_mroot) or os.path.exists(_mroot):
                _sh.rmtree(_mroot, ignore_errors=True) if not os.path.islink(_mroot) else os.unlink(_mroot)
            os.symlink(_ANTE_CANON, _mroot)
        except Exception:
            os.makedirs(_mroot, exist_ok=True)
            for _f in _ANTE_FILES:
                _sh.copy2(os.path.join(_ANTE_CANON, _f), os.path.join(_mroot, _f))
print(f"✅ [antelopev2] 配置完了: {_ANTE_CANON}(+ /root/.insightface・/content/models ミラー)・5 onnx", flush=True)

# ─── Phase 2: torchsde 事前 install ─────────────────────────
for dep in ['torchsde']:
    try:
        __import__(dep)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", dep], check=True)


print("\n🟢 [Cell-run] PHASE 3: CLEAN 配置(writer 射程外 /content/aibo_clean へ展開)", flush=True)
# ============================================================
# 🥷 AIBO 起動 [Cell C] ── CLEAN 配置(writer 射程外パスへ git blob から展開)
#   /content/aibo_src は外部 writer に旧版(24113acf・repo に存在しない blob)へ上書きされ続ける。
#   .git(object store)は無事なので、HEAD blob を cat-file で取り出し /content/aibo_clean に直書き。
#   CLEAN は writer の射程外 → 正版を保持(PO 実証)。clone/checkout は使わない(本 env で rc=0 詐欺)。
# ============================================================
import os, sys, subprocess, shutil, hashlib, json, base64

_SRC, _CLEAN, _BR = "/content/aibo_src", "/content/aibo_clean", "sync/colab"
_CLEAN_URL = "https://github.com/miya390831-a11y/aibo_v8.git"

from google.colab import userdata as _udata
_PAT = _udata.get('GH_PAT')
if not _PAT:
    raise RuntimeError("[Cell C] FATAL: GH_PAT 未設定(Colab Secrets)。clone できません")

# git env 無害化(漏れた GIT_* が git を別 worktree に逃がすのを防止)
for _gv in ("GIT_DIR", "GIT_WORK_TREE", "GIT_INDEX_FILE", "GIT_OBJECT_DIRECTORY",
            "GIT_COMMON_DIR", "GIT_CEILING_DIRECTORIES", "GIT_NAMESPACE"):
    os.environ.pop(_gv, None)

# ── 1) aibo_src を fresh clone(目的は .git=object store の取得。working .py が writer に
#       汚染されても object store は健全 → cat-file は常に正版を出せる)──
shutil.rmtree(_SRC, ignore_errors=True)
subprocess.run(["rm", "-rf", _SRC], check=False)
_auth = base64.b64encode(f"x-access-token:{_PAT}".encode()).decode()   # 値はログに出さない
_hdr = f"http.extraHeader=AUTHORIZATION: Basic {_auth}"
# 注意: argv に base64 PAT が載るため、この clone の argv はログに出さない
_rc = subprocess.run(["git", "-c", _hdr, "clone", "--quiet", "--branch", _BR, _CLEAN_URL, _SRC]).returncode
if _rc != 0:
    raise RuntimeError(f"[Cell C] FATAL: git clone 失敗 rc={_rc}(PAT/権限/network を確認・argv は非表示)")
del _PAT, _auth, _hdr
subprocess.run(["git", "-C", _SRC, "remote", "set-url", "origin", _CLEAN_URL], check=False)
_sha = subprocess.run(["git", "-C", _SRC, "rev-parse", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
print(f"✅ [Cell C] aibo_src fresh clone @ {_sha[:8]}({_BR})", flush=True)

def _cat_blob(_rel):
    _p = subprocess.run(["git", "-C", _SRC, "cat-file", "-p", f"HEAD:{_rel}"], capture_output=True)
    if _p.returncode != 0:
        raise RuntimeError(f"[Cell C] FATAL: cat-file HEAD:{_rel} rc={_p.returncode}: "
                           f"{_p.stderr.decode('utf-8', 'replace')[:200]}")
    return _p.stdout

def _materialize_one(_rel, _root):
    """HEAD:<rel> の blob を CLEAN に直書き。削除→tmp+fsync→atomic replace→読戻し byte 一致 assert。"""
    _blob = _cat_blob(_rel)
    _dst = os.path.join(_root, _rel.replace("/", os.sep))
    os.makedirs(os.path.dirname(_dst) or _root, exist_ok=True)
    if os.path.lexists(_dst):
        os.remove(_dst)
    _tmp = _dst + ".aibo_tmp"
    with open(_tmp, "wb") as _wf:
        _wf.write(_blob)
        _wf.flush()
        os.fsync(_wf.fileno())
    os.replace(_tmp, _dst)
    with open(_dst, "rb") as _rf:
        _got = _rf.read()
    if _got != _blob:
        raise RuntimeError(f"[Cell C] FATAL: 書込みが反映されない(write no-op): {_rel} "
                           f"(disk={len(_got)} != blob={len(_blob)})")
    return len(_blob)

# ── 2) HEAD の全 tracked を CLEAN へ展開(毎回まっさら → 残骸の上に重ねない)──
shutil.rmtree(_CLEAN, ignore_errors=True)
subprocess.run(["rm", "-rf", _CLEAN], check=False)
os.makedirs(_CLEAN, exist_ok=True)
_files = [f for f in subprocess.run(
    ["git", "-C", _SRC, "ls-tree", "-r", "-z", "--name-only", "HEAD"],
    capture_output=True, text=True, encoding="utf-8").stdout.split("\x00") if f]
for _rel in _files:
    _materialize_one(_rel, _CLEAN)
print(f"✅ [Cell C] cat-file→直書き materialize: {len(_files)} files → {_CLEAN}", flush=True)

# ── 3) CLEAN disk md5 == committed manifest を assert(silent fail 禁止)──
def _norm_md5(path):
    with open(path, "rb") as _f:
        _raw = _f.read()
    _t = _raw.decode("utf-8", errors="replace").replace("\r\n", "\n").replace("\r", "\n")
    return hashlib.md5(_t.encode("utf-8")).hexdigest()

with open(os.path.join(_CLEAN, "module_manifest.json"), "r", encoding="utf-8") as _f:
    _man = json.load(_f).get("modules", {})
_bad = []
for _fn, _exp in sorted(_man.items()):
    _fp = os.path.join(_CLEAN, _fn)
    if not os.path.exists(_fp):
        _bad.append(f"{_fn}: MISSING (expected {_exp[:8]})")
        continue
    if _norm_md5(_fp) != _exp:
        _bad.append(f"{_fn}: disk={_norm_md5(_fp)[:8]} != manifest={_exp[:8]}")
if _bad:
    raise RuntimeError("[Cell C] FATAL: CLEAN disk != manifest:\n  " + "\n  ".join(_bad))
print(f"✅ [Cell C] CLEAN disk md5 == manifest({len(_man)} module)・正版を {_CLEAN}(writer 射程外)に配置完了", flush=True)


print("\n🟢 [Cell-run] PHASE 4: CLEAN から import→build→FastAPI/ngrok(PuLID 検証)", flush=True)
# ============================================================
# 🥷 AIBO 起動 [Cell D] ── CLEAN から import→build→FastAPI/ngrok(冪等・PuLID 検証)
#   AIBO_ROOT=/content/aibo_clean。clone/materialize/rmtree はしない(CLEAN を触らない・read-only 検証のみ)。
#   再実行で OOM しないよう、(冪等①)起動済みなら build skip /(冪等②)前回の VRAM を解放してから build。
#   PuLID が縮退していたら GATE 不可なので即 raise(Nika identity 必須)。
# ============================================================
import os, sys, re, gc, time, json, hashlib, importlib, subprocess
import requests as _rq

AIBO_ROOT = "/content/aibo_clean"
assert os.path.isdir(AIBO_ROOT), f"FATAL: {AIBO_ROOT} が無い。Cell C を先に実行すること"
for _k in ("HF_HOME", "TRANSFORMERS_CACHE", "HUGGINGFACE_HUB_CACHE", "HF_HUB_CACHE"):
    os.environ[_k] = "/content/aibo_hf_cache"
_CONSTR = "/content/aibo_constraints.txt"
if os.path.exists(_CONSTR):
    os.environ["PIP_CONSTRAINT"] = _CONSTR        # build 中の install_all にも numpy 下限を効かせる
sys.dont_write_bytecode = True
_clean_real = os.path.realpath(AIBO_ROOT)
print("=" * 60, flush=True)
print(f"🥷 [Cell D] AIBO_ROOT = {AIBO_ROOT}   (CLEAN 起動・transport 無し)", flush=True)
print("=" * 60, flush=True)

# ── ★前提ガード(Cell A の成否に依存しない): PuLID に必須の onnxruntime が import できなければ
#    build/生成に進まない。これが無いと縮退した素 FluxPipeline が起動し、生成 Pass1 で
#    `FluxPipeline.__call__() got 'id_image'` で落ちる(05 は pipe_base 型に関係なく id_image を渡す)。
#    既起動チェックより前に置く = 縮退サーバを「正常」と誤報告しないため。
try:
    import onnxruntime as _ort_pre
    print(f"✅ [Cell D] 前提 onnxruntime import OK (v{_ort_pre.__version__})", flush=True)
except Exception as _e:
    raise RuntimeError(f"[Cell D] FATAL: onnxruntime が import 不可({type(_e).__name__}: {_e})。"
                       "Cell A を先に成功させること(このまま進むと PuLID 縮退で生成 Pass1 が id_image エラーになる)。")

# ── ★前提ガード②: insightface antelopev2 の .onnx が探されうる root に在るか(Cell B で配置)。
#    無いと build 時に PuLID が縮退する真因なので、build 前にここで止める。
_ANTE_FILES = ["1k3d68.onnx", "2d106det.onnx", "genderage.onnx", "glintr100.onnx", "scrfd_10g_bnkps.onnx"]
_ANTE_ROOTS = [
    os.path.join(os.environ.get("HF_HOME", "/content/aibo_hf_cache"), "insightface", "models", "antelopev2"),
    "/root/.insightface/models/antelopev2",
    "/content/models/antelopev2",
]
_ante_ok = [d for d in _ANTE_ROOTS
            if all(os.path.exists(os.path.join(d, f)) and os.path.getsize(os.path.join(d, f)) > 0 for f in _ANTE_FILES)]
if not _ante_ok:
    raise RuntimeError("[Cell D] FATAL: insightface antelopev2 の .onnx がどの root にも無い "
                       f"({_ANTE_ROOTS})。Cell B(モデル warm)を実行して配置すること"
                       "(無いと PuLID が縮退して生成 Pass1 が id_image エラーになる)。")
print(f"✅ [Cell D] 前提 antelopev2 OK: {_ante_ok[0]}", flush=True)

# ── 冪等①: FastAPI が既に生存していれば build skip(VRAM 二重積み OOM を回避)──
def _alive():
    try:
        r = _rq.get("http://localhost:8000/api/system/status", timeout=2)
        return r.status_code == 200 and r.json().get("orchestrator_attached")
    except Exception:
        return False

if _alive():
    b = _rq.get("http://localhost:8000/api/system/status", timeout=5).json()
    print(f"✅ [Cell D] FastAPI 既起動 → build skip(冪等)。GPU={b.get('gpu_name')} "
          f"VRAM={b.get('vram_used_gb')}/{b.get('vram_total_gb')}GB", flush=True)
    _ef = "/content/.env.local.latest.txt"
    if os.path.exists(_ef):
        print(open(_ef).read().strip(), flush=True)
    raise SystemExit("[Cell D] 既起動につき再 build せず終了(冪等)")

# ── 冪等②: 前回の pipeline/VRAM を解放(再実行時の二重積み OOM 回避)──
import torch
for _nm in ("aibo", "orchestrator", "pm", "ie", "ipa", "mod04", "mod07", "tf"):
    if _nm in globals():
        try:
            del globals()[_nm]
        except Exception:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print(f"🧹 [Cell D] VRAM 解放後: allocated={torch.cuda.memory_allocated()/1e9:.1f} GB", flush=True)

# ── sys.path 衛生(先頭を CLEAN に固定・aibo_src 等コア path を除去)──
_CORE_BASENAMES = {
    "01_config.py", "02_colab_setup.py", "03_identity_engine.py", "04_pipeline_manager.py",
    "05_orchestrator.py", "06_ui.py", "07_main.py", "08_face_refiner.py", "09_fastapi_server.py",
    "10_mask_engine.py", "11_pulid_cn_pipeline.py", "15_makeup_engine.py", "16_pose_extractor.py",
    "17_neutralize_engine.py",
}
_kept = []
for _p in sys.path:
    try:
        _rp = os.path.realpath(_p) if _p else _p
    except Exception:
        _kept.append(_p)
        continue
    if _rp == _clean_real:
        continue
    if _p and os.path.isdir(_p) and any(os.path.exists(os.path.join(_p, _b)) for _b in _CORE_BASENAMES):
        print(f"🧹 [Cell D] stale code path を sys.path から除外: {_p}", flush=True)
        continue
    _kept.append(_p)
sys.path[:] = [AIBO_ROOT] + _kept
print(f"✅ [Cell D] sys.path[0] = {sys.path[0]}", flush=True)

# ── sys.modules purge(^\d{2}_ / __file__ が aibo_src 含む他パス / コア basename を一掃)──
def _is_aibo(_n, _m):
    if re.match(r"^\d{2}_", _n):
        return True
    _f = getattr(_m, "__file__", None)
    if not _f:
        return False
    _rf = os.path.realpath(_f)
    return _rf.startswith(_clean_real) or "/aibo_src" in _rf or os.path.basename(_rf) in _CORE_BASENAMES

_killed = [n for n, m in list(sys.modules.items()) if _is_aibo(n, m)]
for _n in _killed:
    del sys.modules[_n]
importlib.invalidate_caches()
print(f"🧹 [Cell D] sys.modules purge: {len(_killed)} module 削除(aibo_src 由来も含め一掃)", flush=True)

# ── CLEAN disk 全 module 正版検証(read-only)──
def _norm_md5(p):
    with open(p, "rb") as _f:
        _raw = _f.read()
    return hashlib.md5(_raw.decode("utf-8", errors="replace").replace("\r\n", "\n").replace("\r", "\n").encode("utf-8")).hexdigest()

with open(os.path.join(AIBO_ROOT, "module_manifest.json"), encoding="utf-8") as _f:
    _man = json.load(_f).get("modules", {})
_db = []
for _fn, _exp in sorted(_man.items()):
    _fp = os.path.join(AIBO_ROOT, _fn)
    if not os.path.exists(_fp):
        _db.append(f"{_fn}: MISSING")
    elif _norm_md5(_fp) != _exp:
        _db.append(f"{_fn}: disk={_norm_md5(_fp)[:8]} != manifest={_exp[:8]}")
if _db:
    raise RuntimeError("[Cell D] FATAL: CLEAN disk != manifest:\n  " + "\n  ".join(_db))
print(f"✅ [Cell D] CLEAN disk md5 == manifest({len(_man)} module)", flush=True)

# ── import(CLEAN から)──
_t = time.perf_counter()
mod01 = importlib.import_module("01_config")
mod02 = importlib.import_module("02_colab_setup")
mod03 = importlib.import_module("03_identity_engine")
mod04 = importlib.import_module("04_pipeline_manager")
mod05 = importlib.import_module("05_orchestrator")
mod06 = importlib.import_module("06_ui")
mod07 = importlib.import_module("07_main")
mod08 = importlib.import_module("08_face_refiner")
print(f"⏱️ import: {time.perf_counter()-_t:.2f}s", flush=True)

# ── runtime md5 == manifest + 全 __file__ が CLEAN 配下 を assert(aibo_src shadow を即露見)──
_mods = {"01_config.py": mod01, "02_colab_setup.py": mod02, "03_identity_engine.py": mod03,
         "04_pipeline_manager.py": mod04, "05_orchestrator.py": mod05, "06_ui.py": mod06,
         "07_main.py": mod07, "08_face_refiner.py": mod08}
_rb = []
for _fn, _m in _mods.items():
    _f = os.path.realpath(getattr(_m, "__file__", "") or "")
    if not _f.startswith(_clean_real):
        _rb.append(f"{_fn}: __file__ が CLEAN 外 = {_f}")
        continue
    _e = _man.get(_fn)
    if _e and _norm_md5(_f) != _e:
        _rb.append(f"{_fn}: runtime={_norm_md5(_f)[:8]} != manifest={_e[:8]}")
if _rb:
    raise RuntimeError("[Cell D] FATAL: runtime md5/__file__ 検証 NG:\n  " + "\n  ".join(_rb))
print(f"✅ [Cell D] runtime md5 == manifest・全 {len(_mods)} core の __file__ が {AIBO_ROOT} 配下", flush=True)
print(f"    03: {_norm_md5(os.path.realpath(mod03.__file__))[:8]} <- {os.path.realpath(mod03.__file__)}", flush=True)

# ── build(AiboMain)──
GenerationConfig = mod01.GenerationConfig
IdentityConfig = mod01.IdentityConfig
AiboMain = mod07.AiboMain
_tb0 = time.perf_counter()
aibo = AiboMain()
if hasattr(aibo, "run"):
    aibo.run(enable_gradio=False)            # A方式運用・Phase G スキップ(07 内 manifest preflight も走る)
print(f"⏱️ AiboMain build: {time.perf_counter()-_tb0:.2f}s", flush=True)
orchestrator = aibo.orchestrator
pm = orchestrator.pm
ie = orchestrator.ie
ipa = ie.ip_adapter
id_cfg = IdentityConfig()

# ── ★PuLID 有効を検証(縮退していたら GATE 不可 → 即 raise)──
_cls = type(pm.pipe_base).__name__
_degraded = getattr(pm, "_pulid_degraded", False)
_has_pulid = hasattr(pm.pipe_base, "pulid_model")
try:
    import onnxruntime as _ort
    _ort_ok = True
except Exception:
    _ort_ok = False
print(f"🔎 [Cell D] PuLID: pipe_base={_cls} / pulid_model={_has_pulid} / onnxruntime={_ort_ok} / degraded={_degraded}", flush=True)
if _degraded or _cls != "PuLIDFluxPipeline" or not _has_pulid:
    raise RuntimeError(f"[Cell D] FATAL: PuLID 縮退(pipe_base={_cls}, onnxruntime={_ort_ok})。"
                       "Cell A の onnxruntime-gpu 導入を確認。Nika identity が出ないため起動中止。")
print("✅ [Cell D] PuLID 有効(PuLIDFluxPipeline・縮退なし)", flush=True)

# ── Phase 1 完成状態セットアップ(本家 Cell 0 と同一)──
tf = pm._shared_transformer
if pm.pipe_cnet is None:
    pm.ensure_controlnet()
pm._cn_forward_wrapped = False
pm._wrap_transformer_forward_for_cn()
if pm.pipe_cnet.image_encoder is None:
    pm.pipe_cnet.image_encoder = ipa._image_encoder
    pm.pipe_cnet.feature_extractor = ipa._feature_extractor
ipa.set_scale(pm.pipe_base, id_cfg.ip_adapter_weight)
ipa.set_scale(pm.pipe_cnet, id_cfg.ip_adapter_weight)
print(f"✅ forward={tf.forward.__qualname__} · IP-Adapter scale={id_cfg.ip_adapter_weight}", flush=True)
print("🎉 Phase 1 完成 · 即生成可能!", flush=True)

# ── FastAPI + ngrok 公開 ──
import traceback as _tbm
try:
    import threading
    for _pkg in ["fastapi", "pyngrok"]:
        try:
            __import__(_pkg)
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])
    from pyngrok import ngrok, conf
    from google.colab import userdata
    srv = importlib.import_module("09_fastapi_server")
    assert os.path.realpath(srv.__file__).startswith(_clean_real), f"09 が CLEAN 外: {srv.__file__}"
    srv.attach_orchestrator(orchestrator, pm)
    threading.Thread(target=lambda: srv.run_server(host="0.0.0.0", port=8000, log_level="warning"),
                     daemon=True, name="aibo-fastapi").start()
    time.sleep(3)
    b = _rq.get("http://localhost:8000/api/system/status", timeout=5).json()
    print(f"✅ FastAPI · GPU={b['gpu_name']} · VRAM={b['vram_used_gb']:.1f}/{b['vram_total_gb']:.1f} GB", flush=True)
    conf.get_default().auth_token = userdata.get('NGROK_AUTH_TOKEN')
    try:
        for _t2 in ngrok.get_tunnels():
            ngrok.disconnect(_t2.public_url)
        ngrok.kill()
        time.sleep(1)
    except Exception:
        pass
    public_url = ngrok.connect(8000, "http").public_url
    time.sleep(3)
    for _i in range(5):
        try:
            if _rq.get(f"{public_url}/api/system/status",
                       headers={"ngrok-skip-browser-warning": "true"}, timeout=10).status_code == 200:
                break
        except Exception:
            pass
        time.sleep(2)
    env_content = f"NEXT_PUBLIC_API_URL={public_url}\nNEXT_PUBLIC_NGROK_SKIP_WARNING=true"
    with open("/content/.env.local.latest.txt", "w") as f:   # CLEAN の .py は触らない → /content に出す
        f.write(env_content)
    print(f"\n🔌 API URL: {public_url}", flush=True)
    print("📋 PC の .env.local にコピー(/content/.env.local.latest.txt にも保存):", flush=True)
    print(env_content, flush=True)
    print("🎉 起動完了(/content/aibo_clean・PuLID 有効)🥷", flush=True)
except Exception as _e:
    print(f"\n❌ serve で例外: {type(_e).__name__}: {_e}", flush=True)
    _tbm.print_exc()
    raise


# ── ログ蛇口 auto-attach(② 配線): [OSS] APPLIED → /content/aibo_gen.log ──────────
#   CLEAN 配下の spigot を exec。root logger + propagate + stdout tee + self-test で確実化。
print("\n🟢 [Cell-run] PHASE 6: ログ蛇口 attach(/content/aibo_gen.log)", flush=True)
try:
    _spig = os.path.join(AIBO_ROOT, "attach_aibo_log_spigot.py")
    if os.path.exists(_spig):
        exec(open(_spig, encoding="utf-8").read())
    else:
        print("⚠️ [Cell-run] 蛇口スクリプトが無い: " + _spig + "(CLEAN materialize を確認)", flush=True)
except Exception as _se:
    print("⚠️ [Cell-run] 蛇口 attach 失敗(serve は成功・継続): "
          + type(_se).__name__ + ": " + str(_se), flush=True)
print("\n🎉 [Cell-run] 完了。localhost:3000 で 1 枚生成 → 確認は下の [任意] セル or "
      "!tail -n 120 /content/aibo_gen.log", flush=True)


In [ ]:
# ============================================================
# 🔍 [任意] 環境確認 (Cell-run 実行後)  ── CLEAN 版
# ============================================================
import os, json, urllib.request

print("=" * 60)
print("🔍 環境確認 (CLEAN 版 · AIBO_ROOT=/content/aibo_clean)")
print("=" * 60)

# 1. FastAPI (local)
try:
    with urllib.request.urlopen("http://127.0.0.1:8000/api/system/status", timeout=10) as resp:
        st = json.loads(resp.read().decode())
    print("✅ localhost:8000")
    print(f"   orchestrator_attached: {st.get('orchestrator_attached')}")
    if st.get("gpu_available"):
        print(f"   GPU: {st.get('gpu_name')}")
        print(f"   VRAM: {st.get('vram_used_gb')}/{st.get('vram_total_gb')} GB ({st.get('vram_pct')}%)")
except Exception as exc:
    print(f"❌ localhost:8000 · {exc}")

# 2. .env.local.latest.txt + ngrok 疎通
_ef = "/content/.env.local.latest.txt"
if os.path.isfile(_ef):
    text = open(_ef, encoding="utf-8").read().strip()
    print(f"\n✅ {_ef}:")
    print(text)
    ngrok_url = None
    for line in text.splitlines():
        if line.startswith("NEXT_PUBLIC_API_URL="):
            ngrok_url = line.split("=", 1)[1].strip()
            break
    if ngrok_url:
        try:
            req = urllib.request.Request(
                f"{ngrok_url.rstrip('/')}/api/system/status",
                headers={"ngrok-skip-browser-warning": "true"},
            )
            with urllib.request.urlopen(req, timeout=15) as resp:
                ext = json.loads(resp.read().decode())
            print(f"\n✅ ngrok 疎通 OK · orchestrator_attached={ext.get('orchestrator_attached')}")
        except Exception as exc:
            print(f"\n⚠️ ngrok 疎通: {exc}")
else:
    print(f"\n⚠️ {_ef} が無い (Cell D を先に実行)")


In [ ]:
# ============================================================
# 🥷 [任意・GATE① 緑の後に PO 実行] ④a guidance A/B 比較(4.0 vs 3.5)
#   同条件(N=14・seed 固定・Nika refs)で guidance 4.0 と 3.5 を 1 枚ずつ生成。
#   原寸保存 + 横並び GRID を出す → PO が肌/identity を見て本番既定を 1 つ選ぶ。
#   prod 非改変(srv._build_configs + orch.generate を呼ぶだけ)。core .py は触らない。
# ============================================================
import os, sys, time
from importlib import import_module
from PIL import Image

AIBO_ROOT = "/content/aibo_clean"
if AIBO_ROOT not in sys.path:
    sys.path.insert(0, AIBO_ROOT)

# ── 比較条件(必要なら PO 調整)──
GUIDANCES = [4.0, 3.5]          # A=本番 UI 既定 / B=EXP-033
SEED      = 12345              # 固定(guidance だけを単離)
N_STEPS   = 14
W, H      = 1024, 1536
PROMPT = ("close-up portrait photo of a young woman, soft natural daylight, "
          "looking directly at camera, 85mm lens, photorealistic, sharp focus on face")
NEG = None

# ── Nika refs(/content/drive/MyDrive/顔 の公式6枚・差し替え可)──
FACE_DIR = "/content/drive/MyDrive/顔"
REF_FILES = [
    "Screenshot_20260414-124011.png", "Screenshot_20260414-124040.png",
    "Screenshot_20260414-124119.png", "Screenshot_20260414-124130.png",
    "Screenshot_20260414-124144.png", "Screenshot_20260510-150734.png",
]
OUT_DIR = "/content/drive/MyDrive/aibo_v7/experiments"
os.makedirs(OUT_DIR, exist_ok=True)

srv = import_module("09_fastapi_server")
pm04 = import_module("04_pipeline_manager")
orch = srv.get_orchestrator()
assert orch is not None, "[④a][STOP] orchestrator 未 attach。先に [Cell-run] を実行"
StudioMode = srv._resolve_studio_mode()

_ref_paths = [os.path.join(FACE_DIR, f) for f in REF_FILES]
_missing = [p for p in _ref_paths if not os.path.exists(p)]
assert not _missing, "[④a][STOP] ref 不足(フォールバックしない): " + str(_missing)
refs = [Image.open(p).convert("RGB") for p in _ref_paths]

def _gen(guidance, seed, save_path, label):
    gen_cfg, id_cfg = srv._build_configs(
        face_imgs=refs, prompt=PROMPT, negative_prompt=NEG,
        num_inference_steps=N_STEPS, guidance_scale=float(guidance),
        seed=int(seed), width=W, height=H,
    )
    print("[④a] ▶ " + label + ": guidance=" + str(guidance) + " seed=" + str(seed)
          + " N=" + str(N_STEPS) + " 生成中…", flush=True)
    t0 = time.perf_counter()
    r = orch.generate(gen_cfg, id_cfg, mode=StudioMode.PORTRAIT, save=False)
    wall = time.perf_counter() - t0
    assert getattr(r, "error", None) is None, "[④a][STOP] " + label + " 失敗: " + str(r.error)
    assert getattr(r, "final_image", None) is not None, "[④a][STOP] " + label + " final_image=None"
    img = r.final_image.convert("RGB")
    img.save(save_path)
    print("[④a] 保存: " + save_path + "  (" + ("%.1f" % wall) + "s · pulid_used="
          + str(getattr(r, "pulid_used", None)) + ")", flush=True)
    return img, wall

print("[④a] warm-up(1回捨て)…", flush=True)
_gen(GUIDANCES[0], SEED, os.path.join(OUT_DIR, "_warm.png"), "WARM")

panels, saved = [], []
for g in GUIDANCES:
    p = os.path.join(OUT_DIR, "guidance_cmp_g" + str(g).replace(".", "_") + "_seed" + str(SEED) + ".png")
    img, wall = _gen(g, SEED, p, "guidance=" + str(g))
    panels.append(("guidance=" + str(g) + " (seed" + str(SEED) + ", " + ("%.1f" % wall) + "s)", img))
    saved.append(p)

GRID = os.path.join(OUT_DIR, "guidance_cmp_GRID_seed" + str(SEED) + ".png")
try:
    pm04._depth_grid(panels, GRID, ncols=len(panels), show=True,
                     title="guidance A/B: 肌(毛穴/産毛/grain)+ Nika identity 保持 を比較(PO が 1 つ選ぶ)")
    print("\n[④a] ✅ GRID: " + GRID, flush=True)
except Exception as _e:
    print("[④a] (warn) GRID 生成失敗(原寸は保存済・継続): " + type(_e).__name__ + ": " + str(_e), flush=True)

print("[④a] 原寸: " + str(saved), flush=True)
print("[④a] PO 手順: 上の GRID / 原寸を見て 4.0 か 3.5 を選択。", flush=True)
print("  選んだ値を本番既定に → frontend portrait-mode.tsx の useState(<guidance>) を変更", flush=True)
print("  (backend 既定 = 01_config.py GenerationConfig.guidance_scale=3.5。UI から渡る値が優先)", flush=True)


In [ ]:
# ============================================================
# 🥷 [任意・guidance 確定後に PO 実行] ④b GATE③ 原寸目視(N=14 × 複数 seed)
#   ④a で確定した guidance で N=14 を複数 seed 生成 → 原寸保存 + GRID。
#   PO が原寸で「D20級肌 + Nika identity 保持」を複数画で目視(機械判定しない)。
# ============================================================
import os, sys, time
from importlib import import_module
from PIL import Image

AIBO_ROOT = "/content/aibo_clean"
if AIBO_ROOT not in sys.path:
    sys.path.insert(0, AIBO_ROOT)

GUIDANCE = 4.0          # ★④a で選んだ確定値に設定(既定は本番 UI 既定 4.0)
N_STEPS  = 14
SEEDS    = [12345, 2024, 7777, 31337]
W, H     = 1024, 1536
PROMPT = ("close-up portrait photo of a young woman, soft natural daylight, "
          "looking directly at camera, 85mm lens, photorealistic, sharp focus on face")
FACE_DIR = "/content/drive/MyDrive/顔"
REF_FILES = [
    "Screenshot_20260414-124011.png", "Screenshot_20260414-124040.png",
    "Screenshot_20260414-124119.png", "Screenshot_20260414-124130.png",
    "Screenshot_20260414-124144.png", "Screenshot_20260510-150734.png",
]
OUT_DIR = "/content/drive/MyDrive/aibo_v7/experiments/gate3"
os.makedirs(OUT_DIR, exist_ok=True)

srv = import_module("09_fastapi_server")
pm04 = import_module("04_pipeline_manager")
orch = srv.get_orchestrator()
assert orch is not None, "[④b][STOP] orchestrator 未 attach。先に [Cell-run] を実行"
StudioMode = srv._resolve_studio_mode()

_ref_paths = [os.path.join(FACE_DIR, f) for f in REF_FILES]
_missing = [p for p in _ref_paths if not os.path.exists(p)]
assert not _missing, "[④b][STOP] ref 不足: " + str(_missing)
refs = [Image.open(p).convert("RGB") for p in _ref_paths]

def _gen(seed, save_path, label):
    gen_cfg, id_cfg = srv._build_configs(
        face_imgs=refs, prompt=PROMPT, negative_prompt=None,
        num_inference_steps=N_STEPS, guidance_scale=float(GUIDANCE),
        seed=int(seed), width=W, height=H,
    )
    print("[④b] ▶ " + label + ": guidance=" + str(GUIDANCE) + " seed=" + str(seed) + " 生成中…", flush=True)
    t0 = time.perf_counter()
    r = orch.generate(gen_cfg, id_cfg, mode=StudioMode.PORTRAIT, save=False)
    wall = time.perf_counter() - t0
    assert getattr(r, "error", None) is None, "[④b][STOP] " + label + " 失敗: " + str(r.error)
    assert getattr(r, "final_image", None) is not None, "[④b][STOP] " + label + " final_image=None"
    img = r.final_image.convert("RGB")
    img.save(save_path)
    print("[④b] 保存: " + save_path + "  (" + ("%.1f" % wall) + "s · pulid_used="
          + str(getattr(r, "pulid_used", None)) + ")", flush=True)
    return img, wall

print("[④b] warm-up(1回捨て)…", flush=True)
_gen(SEEDS[0], os.path.join(OUT_DIR, "_warm.png"), "WARM")

panels, saved = [], []
for sd in SEEDS:
    p = os.path.join(OUT_DIR, "gate3_g" + str(GUIDANCE).replace(".", "_") + "_seed" + str(sd) + ".png")
    img, wall = _gen(sd, p, "seed" + str(sd))
    panels.append(("seed" + str(sd) + " (" + ("%.1f" % wall) + "s)", img))
    saved.append(p)

GRID = os.path.join(OUT_DIR, "gate3_GRID_g" + str(GUIDANCE).replace(".", "_") + ".png")
try:
    pm04._depth_grid(panels, GRID, ncols=len(panels), show=True,
                     title="GATE③ (guidance=" + str(GUIDANCE) + "): PO 原寸目視 — D20級肌 + Nika identity 保持 ×複数seed")
    print("\n[④b] ✅ GRID: " + GRID, flush=True)
except Exception as _e:
    print("[④b] (warn) GRID 生成失敗(原寸は保存済・継続): " + type(_e).__name__ + ": " + str(_e), flush=True)

print("[④b] 原寸(PO は各ファイルを原寸で開いて目視):", flush=True)
for _p in saved:
    print("   " + _p, flush=True)
print("[④b] GATE③ 判定は PO 目視(機械判定しない)。OK なら guidance " + str(GUIDANCE) + " を本番確定。", flush=True)
